In [243]:
import pandas as pd 
import numpy as np

In [244]:
# read data 
df1 = pd.read_parquet("cleaned1.parquet")
df2 = pd.read_parquet("cleaned2.parquet")
df3 = pd.read_parquet("cleaned3.parquet")

In [245]:
df1.columns

Index(['Zone_name', 'Datetime', 'Zone_temp', 'Slab_temp', 'Dew_temp',
       'Ambient_temp', 'Damper_status', 'Fan_status', 'Datetime_diff_mins',
       'Date', 'Time', 'Year', 'Month', 'Day', 'DOW', 'Season',
       'Zone_temp_diff', 'Slab_temp_diff', 'Dew_temp_diff',
       'Ambient_temp_diff', 'Cumulative_fan_on_mins', 'Fan_on_group',
       'Cumulative_damper_open_mins', 'Damper_open_group', 'building_no',
       'Faulty'],
      dtype='object')

In [246]:
df1c = df1.copy()
df2c = df2.copy()
df3c = df3.copy()

### Data Cleaning

In [247]:
df1 = df1[df1['Faulty'] == False]
df2 = df2[df2['Faulty'] == False]
df3 = df3[df3['Faulty'] == False]

In [248]:
# Data Cleanings
drop_columns_common = ["building_no", "Fan_on_group", "Cumulative_fan_on_mins", "Date", "Time", "Year", "Cumulative_damper_open_mins"]
drop_columns_df1 = drop_columns_common + ["Slab_temp", "Dew_temp", "Slab_temp_diff", "Dew_temp_diff", "Damper_open_group"]
drop_columns_df2 = drop_columns_common
drop_columns_df3 = drop_columns_common + ["Damper_open_group", "Louver_open_group", "Cumulative_louver_open_mins"]

In [249]:
# Conditional Dropping = creates a list of columns to drop
df1 = df1.drop(columns=[col for col in drop_columns_df1 if col in df1.columns])
df2 = df2.drop(columns=[col for col in drop_columns_df2 if col in df2.columns]) 
df3 = df3.drop(columns=[col for col in drop_columns_df3 if col in df3.columns])

In [250]:
# Extracting DOY from the Datetime Column; creates new column Day_of_year in each DF
df1['Day_of_Year'] = df1['Datetime'].dt.dayofyear
df2['Day_of_Year'] = df2['Datetime'].dt.dayofyear
df3['Day_of_Year'] = df3['Datetime'].dt.dayofyear

In [251]:
# creates new col Minutes_Past_midnight in all df that represents number of minutes have passed since midnight
df1['Minutes_Past_Midnight'] = df1['Datetime'].dt.hour * 60 + df1['Datetime'].dt.minute
df2['Minutes_Past_Midnight'] = df2['Datetime'].dt.hour * 60 + df2['Datetime'].dt.minute
df3['Minutes_Past_Midnight'] = df3['Datetime'].dt.hour * 60 + df3['Datetime'].dt.minute

In [252]:
df1.columns

Index(['Zone_name', 'Datetime', 'Zone_temp', 'Ambient_temp', 'Damper_status',
       'Fan_status', 'Datetime_diff_mins', 'Month', 'Day', 'DOW', 'Season',
       'Zone_temp_diff', 'Ambient_temp_diff', 'Faulty', 'Day_of_Year',
       'Minutes_Past_Midnight'],
      dtype='object')

In [253]:
for df in [df1, df2, df3]:
    ints = [i for i in range(len(df.Zone_name.unique()))]
    for i, zone in zip(ints, df.Zone_name.unique()):
        df.loc[df.Zone_name==zone,'Zone_name'] = i
    df.Zone_name = df.Zone_name.astype(int)
    df['Year'] = df.Datetime.dt.year
    
df1.Fan_status = df1.Fan_status.apply(lambda x: 0 if x=='Off' else 1 if x=='On' else np.nan).astype(float)
df2.Fan_status = df2.Fan_status.apply(lambda x: 0 if x=='Off' else 1 if x=='On' else np.nan).astype(float)
df3.Fan_status = df3.Fan_status.apply(lambda x: 0 if x=='Off' else 1 if x=='On' else np.nan).astype(float)

df1 = df1[[col for col in df1 if df1[col].dtype in [int, float, bool]] + ['Season']]
df2 = df2[[col for col in df2 if df2[col].dtype in [int, float, bool]] + ['Season']]
df3 = df3[[col for col in df3 if df3[col].dtype in [int, float, bool]] + ['Season']]

In [254]:
df1.columns

Index(['Zone_name', 'Zone_temp', 'Ambient_temp', 'Damper_status', 'Fan_status',
       'Datetime_diff_mins', 'Month', 'Day', 'DOW', 'Zone_temp_diff',
       'Ambient_temp_diff', 'Faulty', 'Day_of_Year', 'Minutes_Past_Midnight',
       'Year', 'Season'],
      dtype='object')

In [114]:
df1.columns

Index(['Zone_name', 'Zone_temp', 'Ambient_temp', 'Damper_status', 'Fan_status',
       'Datetime_diff_mins', 'Month', 'Day', 'DOW', 'Zone_temp_diff',
       'Ambient_temp_diff', 'Faulty', 'Day_of_Year', 'Minutes_Past_Midnight',
       'Year', 'Season'],
      dtype='object')

### Additional processing for buildings required for deep learning

In [255]:
# fill NaN values with mean
# building 1
fill_columns1 = ['Ambient_temp','Damper_status','Ambient_temp_diff']
for col in fill_columns1:
    df1[col] = df1[col].fillna(df1[col].mean())
# building 2
fill_columns2 = ["Zone_c02","Ambient_temp_diff"]
for col in fill_columns2:
    df2[col] = df2[col].fillna(df2[col].mean())
# building 3
fill_columns3 = ["Datetime_diff_mins","Zone_temp_diff","Slab_temp_diff","Dew_temp_diff","Ambient_temp_diff"]
for col in fill_columns3:
    df3[col] = df3[col].fillna(df3[col].mean())

C:\Users\tousi\AppData\Local\Temp\ipykernel_35192\3724472727.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3[col] = df3[col].fillna(df3[col].mean())


In [256]:
df1_encoded = df1.copy()
df2_encoded = df2.copy()
df3_encoded = df3.copy()

In [257]:
# one-hot encode for categorical data
from sklearn.preprocessing import OneHotEncoder
# building 1
encoder1 = OneHotEncoder(sparse_output=False)
encoded1_array = encoder1.fit_transform(df1_encoded[['Zone_name','Season']])
encoded_df1 = pd.DataFrame(encoded1_array,columns=encoder1.get_feature_names_out(['Zone_name', 'Season']))
df1_encoded = df1_encoded.drop(columns=['Zone_name','Season'])
# building 2
encoder2 = OneHotEncoder(sparse_output=False)
encoded2_array = encoder2.fit_transform(df2_encoded[['Zone_name','Season']])
encoded_df2 = pd.DataFrame(encoded2_array,columns=encoder2.get_feature_names_out(['Zone_name', 'Season']))
df2_encoded = df2_encoded.drop(columns=['Zone_name','Season'])
# building 3
encoder3 = OneHotEncoder(sparse_output=False)
encoded3_array = encoder3.fit_transform(df3[['Zone_name','Season']])
encoded_df3 = pd.DataFrame(encoded3_array,columns=encoder3.get_feature_names_out(['Zone_name','Season']))
df3_encoded = df3_encoded.drop(columns=['Zone_name','Season'])

In [258]:
import pickle
with open("onehot_encoder_building1","wb") as file:
    pickle.dump(encoder1,file)
with open("onehot_encoder_building2","wb") as file:
    pickle.dump(encoder2,file)
with open("onehot_encoder_building3","wb") as file:
    pickle.dump(encoder3,file)

In [259]:
# merge encoded dataset
df1_encoded = pd.concat([df1_encoded,encoded_df1],axis = 1)
df2_encoded = pd.concat([df2_encoded,encoded_df2],axis = 1)
df3_encoded = pd.concat([df3_encoded,encoded_df3],axis = 1)

In [260]:
# drop faulty column
df1_encoded = df1_encoded.drop(columns=['Faulty'])
df2_encoded = df2_encoded.drop(columns=['Faulty'])
df3_encoded = df3_encoded.drop(columns=['Faulty'])

In [261]:
# minmax scale non-categorical data
from sklearn.preprocessing import MinMaxScaler
# building 1
scale1_columns = ['Zone_temp', 'Ambient_temp', 'Damper_status','Datetime_diff_mins', 'Zone_temp_diff', 'Ambient_temp_diff']
scaler1 = MinMaxScaler()
scaled1_data = scaler1.fit_transform(df1_encoded[scale1_columns])
scaled1_df = pd.DataFrame(scaled1_data,columns=scale1_columns)
# building 2
scaler2 = MinMaxScaler()
scale2_columns = ['Zone_temp', 'Slab_temp', 'Dew_temp', 'Ambient_temp', 'Zone_c02', 'Datetime_diff_mins', 'Zone_temp_diff', 'Slab_temp_diff',
       'Dew_temp_diff', 'Ambient_temp_diff']
scaled2_data = scaler2.fit_transform(df2_encoded[scale2_columns])
scaled2_df = pd.DataFrame(scaled2_data,columns=scale2_columns)
# building 3
scaler3 = MinMaxScaler()
scale3_columns = ['Zone_temp', 'Slab_temp', 'Dew_temp', 'Ambient_temp', 'Zone_c02',
       'Damper_status', 'Datetime_diff_mins', 'Zone_temp_diff',
       'Slab_temp_diff', 'Dew_temp_diff', 'Ambient_temp_diff']
scaled3_data = scaler3.fit_transform(df3_encoded[scale3_columns])
scaled3_df = pd.DataFrame(scaled3_data,columns=scale3_columns)


In [262]:
with open("minmax_building1","wb") as file:
    pickle.dump(scaler1,file)
with open("minmax_building2","wb") as file:
    pickle.dump(scaler2,file)
with open("minmax_building3","wb") as file:
    pickle.dump(scaler3,file)

In [263]:
# drop columns which were scaled using minmax scaler
df1_encoded = df1_encoded.drop(columns=scale1_columns)
df2_encoded = df2_encoded.drop(columns=scale2_columns)
df3_encoded = df3_encoded.drop(columns=scale3_columns)

In [264]:
df1_encoded.isna().sum()

Fan_status               292552
Month                    292552
Day                      292552
DOW                      292552
Day_of_Year              292552
Minutes_Past_Midnight    292552
Year                     292552
Zone_name_0              292552
Zone_name_1              292552
Zone_name_2              292552
Zone_name_3              292552
Zone_name_4              292552
Zone_name_5              292552
Zone_name_6              292552
Zone_name_7              292552
Zone_name_8              292552
Zone_name_9              292552
Zone_name_10             292552
Zone_name_11             292552
Zone_name_12             292552
Zone_name_13             292552
Zone_name_14             292552
Zone_name_15             292552
Zone_name_16             292552
Zone_name_17             292552
Zone_name_18             292552
Zone_name_19             292552
Zone_name_20             292552
Zone_name_21             292552
Season_1                 292552
Season_2                 292552
Season_3

In [265]:
# merge one-hot encoded data and minmax scaled data together
df1_encoded = pd.concat([df1_encoded,scaled1_df],axis=1)
df2_encoded = pd.concat([df2_encoded,scaled2_df],axis=1)
df3_encoded = pd.concat([df3_encoded,scaled3_df],axis=1)
print(len(df1_encoded.columns))
print(df1_encoded.columns)
print(len(df2_encoded.columns))
print(df2_encoded.columns)
print(len(df3_encoded.columns))
print(df3_encoded.columns)

39
Index(['Fan_status', 'Month', 'Day', 'DOW', 'Day_of_Year',
       'Minutes_Past_Midnight', 'Year', 'Zone_name_0', 'Zone_name_1',
       'Zone_name_2', 'Zone_name_3', 'Zone_name_4', 'Zone_name_5',
       'Zone_name_6', 'Zone_name_7', 'Zone_name_8', 'Zone_name_9',
       'Zone_name_10', 'Zone_name_11', 'Zone_name_12', 'Zone_name_13',
       'Zone_name_14', 'Zone_name_15', 'Zone_name_16', 'Zone_name_17',
       'Zone_name_18', 'Zone_name_19', 'Zone_name_20', 'Zone_name_21',
       'Season_1', 'Season_2', 'Season_3', 'Season_4', 'Zone_temp',
       'Ambient_temp', 'Damper_status', 'Datetime_diff_mins', 'Zone_temp_diff',
       'Ambient_temp_diff'],
      dtype='object')
22
Index(['Fan_status', 'Month', 'Day', 'DOW', 'Day_of_Year',
       'Minutes_Past_Midnight', 'Year', 'Zone_name_0', 'Season_1', 'Season_2',
       'Season_3', 'Season_4', 'Zone_temp', 'Slab_temp', 'Dew_temp',
       'Ambient_temp', 'Zone_c02', 'Datetime_diff_mins', 'Zone_temp_diff',
       'Slab_temp_diff', 'Dew_temp_di

In [266]:
# drop any rows which have null values
df1_encoded = df1_encoded.dropna(subset=df1_encoded.columns)
df2_encoded = df2_encoded.dropna(subset=df2_encoded.columns)
df3_encoded = df3_encoded.dropna(subset=df3_encoded.columns)

In [272]:
df1_encoded.columns

Index(['Fan_status', 'Month', 'Day', 'DOW', 'Day_of_Year',
       'Minutes_Past_Midnight', 'Year', 'Zone_name_0', 'Zone_name_1',
       'Zone_name_2', 'Zone_name_3', 'Zone_name_4', 'Zone_name_5',
       'Zone_name_6', 'Zone_name_7', 'Zone_name_8', 'Zone_name_9',
       'Zone_name_10', 'Zone_name_11', 'Zone_name_12', 'Zone_name_13',
       'Zone_name_14', 'Zone_name_15', 'Zone_name_16', 'Zone_name_17',
       'Zone_name_18', 'Zone_name_19', 'Zone_name_20', 'Zone_name_21',
       'Season_1', 'Season_2', 'Season_3', 'Season_4', 'Zone_temp',
       'Ambient_temp', 'Damper_status', 'Datetime_diff_mins', 'Zone_temp_diff',
       'Ambient_temp_diff'],
      dtype='object')

In [271]:
print(len(df1_encoded.columns))

39


In [273]:
df1_encoded = df1_encoded.drop(columns=['Month','Day','DOW','Day_of_Year','Year','Minutes_Past_Midnight'])
df2_encoded = df2_encoded.drop(columns=['Month','Day','DOW','Day_of_Year','Year','Minutes_Past_Midnight'])
df3_encoded = df3_encoded.drop(columns=['Month','Day','DOW','Day_of_Year','Year','Minutes_Past_Midnight'])

In [274]:
# retrieve feature columns
feature1_col = [col for col in df1_encoded.columns if col != 'Fan_status']
feature2_col = [col for col in df2_encoded.columns if col != "Fan_status"]
feature3_col = [col for col in df3_encoded.columns if col != "Fan_status"]
print(len(feature1_col))
print(len(feature2_col))
print(len(feature3_col))

32
15
30


### Create training/testing datasets for each buildings

In [279]:
# define feature and target 
# building 1
model1_X = df1_encoded[feature1_col]
model1_y = df1_encoded['Fan_status']
# building 2
model2_X = df2_encoded[feature2_col]
model2_y = df2_encoded['Fan_status']
# building 3
model3_X = df3_encoded[feature3_col]
model3_y = df3_encoded['Fan_status']

In [280]:
model2_X.head()

,Zone_name_0,Season_1,Season_2,Season_3,Season_4,Zone_temp,Slab_temp,Dew_temp,Ambient_temp,Zone_c02,Datetime_diff_mins,Zone_temp_diff,Slab_temp_diff,Dew_temp_diff,Ambient_temp_diff
2341,1.0,0.0,1.0,0.0,0.0,0.242502,0.551346,0.581882,0.267442,0.151282,0.000058,0.583488,0.674667,0.519310,0.506757
2342,1.0,0.0,1.0,0.0,0.0,0.246331,0.551346,0.577526,0.264535,0.165477,0.000058,0.584416,0.674667,0.511093,0.500000
2343,1.0,0.0,1.0,0.0,0.0,0.264837,0.550349,0.577526,0.264535,0.159141,0.000058,0.605751,0.673333,0.519310,0.506757
2344,1.0,0.0,1.0,0.0,0.0,0.255265,0.550349,0.577526,0.264535,0.150132,0.000058,0.564935,0.674667,0.519310,0.506757
2345,1.0,0.0,1.0,0.0,0.0,0.248245,0.549352,0.573171,0.261628,0.155907,0.000058,0.568646,0.673333,0.511093,0.500000


In [281]:
# convert data to tensors so that they can be processed in PyTorch library
import torch
# building 1
model1_X_tensor = torch.tensor(model1_X.values, dtype=torch.float32)
model1_y_tensor = torch.tensor(model1_y.values, dtype=torch.long)
# building 2
model2_X_tensor = torch.tensor(model2_X.values, dtype=torch.float32)
model2_y_tensor = torch.tensor(model2_y.values, dtype=torch.long)
# building 3
model3_X_tensor = torch.tensor(model3_X.values, dtype=torch.float32)
model3_y_tensor = torch.tensor(model3_y.values, dtype=torch.long)

In [282]:
print(model1_X_tensor.size())
print(model2_X_tensor.size())
print(model3_X_tensor.size())

torch.Size([84269, 32])
torch.Size([17447, 15])
torch.Size([323109, 30])


In [283]:
# split dataset into training and test sets
from sklearn.model_selection import train_test_split
# building 1
model1_X_train, model1_X_test, model1_y_train, model1_y_test = train_test_split(model1_X_tensor,model1_y_tensor,test_size=0.3)
# building 2
model2_X_train, model2_X_test, model2_y_train, model2_y_test = train_test_split(model2_X_tensor,model2_y_tensor,test_size=0.3)
# building 3
model3_X_train, model3_X_test, model3_y_train, model3_y_test = train_test_split(model3_X_tensor,model3_y_tensor,test_size=0.3)

In [284]:
# Define customized data loading class
from torch.utils.data import Dataset
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels 
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [285]:
# create datasets 
# building 1
model1_train_dataset = CustomDataset(model1_X_train,model1_y_train)
model1_test_dataset = CustomDataset(model1_X_test,model1_y_test)
# building 2
model2_train_dataset = CustomDataset(model2_X_train,model2_y_train)
model2_test_dataset = CustomDataset(model2_X_test,model2_y_test)
# building 1
model3_train_dataset = CustomDataset(model3_X_train,model3_y_train)
model3_test_dataset = CustomDataset(model3_X_test,model3_y_test)

In [286]:
# create dataloader object for each building
from torch.utils.data import DataLoader
# building 1
train1_dataloader = DataLoader(model1_train_dataset,batch_size=128,shuffle=True)
test1_dataloader = DataLoader(model1_train_dataset,batch_size=128,shuffle=True)
# building 2
train2_dataloader = DataLoader(model2_train_dataset,batch_size=32,shuffle=True)
test2_dataloader = DataLoader(model2_train_dataset,batch_size=32,shuffle=True)
# building 3
train3_dataloader = DataLoader(model3_train_dataset,batch_size=128,shuffle=True)
test3_dataloader = DataLoader(model3_train_dataset,batch_size=128,shuffle=True)

### Build deep learning model for each building

In [287]:
# define model for each building
from torch import nn
# feed-forward network for building 1
class building1_FFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(32,20)
        self.hidden2 = nn.Linear(20,12)
        self.hidden3 = nn.Linear(12,6)
        self.output = nn.Linear(6,2)
    def forward(self,x):
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        x = torch.relu(self.hidden3(x))
        x = self.output(x)
        return x
    
# feed-forward network for building 2
class building2_FFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(15,10)
        self.hidden2 = nn.Linear(10,5)
        self.output = nn.Linear(5,2)
    def forward(self,x):
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        x = self.output(x)
        return x
    
# feed-forward network for building 3
class building3_FFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(30,18)
        self.hidden2 = nn.Linear(18,10)
        self.hidden3 = nn.Linear(10,6)
        self.output = nn.Linear(6,2)
    def forward(self,x):
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        x = torch.relu(self.hidden3(x))
        x = self.output(x)
        return x

In [288]:
# define train loop function
def train_loop(model, dataloader, optimiser, loss_fn):
    model.train()
    for batch, (X,y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred,y)
        loss.backward()
        optimiser.step()
        optimiser.zero_grad()

        if batch % 100 == 0:
            print(f"Batch {batch}: {loss}")

# define test loop function
def test_loop(model,dataloader,loss_fn):
    model.eval()
    num_batches = len(dataloader)
    y_pred, y_true = [], []
    with torch.no_grad():
        total = 0
        correct = 0
        loss = 0
        for X, y in dataloader:
            pred = model(X)
            loss += loss_fn(pred,y).item()
            predicted_classes = pred.argmax(dim=1)
            correct += (predicted_classes == y).sum().item()
            total += y.size(0)
            # f1_func.update(predicted_classes,y)
    # f1_score = f1_func.compute()
    accuracy = (correct/total) * 100
    loss /= num_batches
    
    print(f"Accuracy: {round(accuracy,3)}, loss: {round(loss,3)}")
    acc = round(accuracy,3)
    return acc


In [289]:
# building 1 model parameters
model1 = building1_FFN()
learning_rate = 0.01 
epoch = 10
optimizer = torch.optim.Adam(model1.parameters(),lr=learning_rate)
loss = torch.nn.CrossEntropyLoss()

In [290]:
accuracy1 = []
# loop to train and test model for building 1
for i in range(epoch):
    ep = i + 1
    print(f"Epoch {i+1}")
    train_loop(model1,train1_dataloader,optimizer,loss)
    acc = test_loop(model1,test1_dataloader,loss)
    accuracy1.append([ep,acc])
print("Done")

Epoch 1
Batch 0: 0.5876188278198242
Batch 100: 0.6271072030067444
Batch 200: 0.657305896282196
Batch 300: 0.6210868954658508
Batch 400: 0.6391562223434448
Accuracy: 69.056, loss: 0.619
Epoch 2
Batch 0: 0.6147892475128174
Batch 100: 0.5810720920562744
Batch 200: 0.57852703332901
Batch 300: 0.6458028554916382
Batch 400: 0.6086748242378235
Accuracy: 69.056, loss: 0.619
Epoch 3
Batch 0: 0.6340013146400452
Batch 100: 0.5454262495040894
Batch 200: 0.5952032804489136
Batch 300: 0.6030060648918152
Batch 400: 0.621647834777832
Accuracy: 69.056, loss: 0.619
Epoch 4
Batch 0: 0.651371955871582
Batch 100: 0.6331062912940979
Batch 200: 0.5878636837005615
Batch 300: 0.6519193649291992
Batch 400: 0.6086635589599609
Accuracy: 69.056, loss: 0.619
Epoch 5
Batch 0: 0.5677428245544434
Batch 100: 0.6212072968482971
Batch 200: 0.6399461030960083
Batch 300: 0.6212894320487976
Batch 400: 0.5877787470817566
Accuracy: 69.056, loss: 0.619
Epoch 6
Batch 0: 0.6212570071220398
Batch 100: 0.6035477519035339
Batch 200

In [291]:
print(accuracy1)

[[1, 69.056], [2, 69.056], [3, 69.056], [4, 69.056], [5, 69.056], [6, 69.056], [7, 69.056], [8, 69.056], [9, 69.056], [10, 69.056]]


In [292]:
# building 2 model parameters
model2 = building2_FFN()
learning_rate = 0.01 
epoch = 10
optimizer = torch.optim.Adam(model2.parameters(),lr=learning_rate)
loss = torch.nn.CrossEntropyLoss()

In [293]:
accuracy2 = []
# loop to train and test model for building 2
for i in range(epoch):
    ep = i + 1
    print(f"Epoch {i+1}")
    train_loop(model2,train2_dataloader,optimizer,loss)
    acc = test_loop(model2,test2_dataloader,loss)
    accuracy2.append([ep,acc])
print("Done")

Epoch 1
Batch 0: 0.7252606153488159
Batch 100: 0.2431507408618927
Batch 200: 0.2517589032649994
Batch 300: 0.030506988987326622
Accuracy: 95.717, loss: 0.171
Epoch 2
Batch 0: 0.12705616652965546
Batch 100: 0.2265777587890625
Batch 200: 0.14046861231327057
Batch 300: 0.1273217797279358
Accuracy: 95.717, loss: 0.171
Epoch 3
Batch 0: 0.13120882213115692
Batch 100: 0.4366886019706726
Batch 200: 0.13435795903205872
Batch 300: 0.24002090096473694
Accuracy: 95.717, loss: 0.17
Epoch 4
Batch 0: 0.1339888870716095
Batch 100: 0.12667801976203918
Batch 200: 0.1498781144618988
Batch 300: 0.05734401196241379
Accuracy: 95.717, loss: 0.171
Epoch 5
Batch 0: 0.31366804242134094
Batch 100: 0.31214866042137146
Batch 200: 0.12223067879676819
Batch 300: 0.07103604078292847
Accuracy: 95.717, loss: 0.17
Epoch 6
Batch 0: 0.03975040465593338
Batch 100: 0.12471312284469604
Batch 200: 0.05639715492725372
Batch 300: 0.12333717197179794
Accuracy: 95.717, loss: 0.176
Epoch 7
Batch 0: 0.4264831840991974
Batch 100: 0.

In [294]:
# building 3 model parameters
model3 = building3_FFN()
learning_rate = 0.01 
epoch = 10
optimizer = torch.optim.Adam(model3.parameters(),lr=learning_rate)
loss = torch.nn.CrossEntropyLoss()

In [295]:
accuracy3 = []
# loop to train and test model for building 3
for i in range(epoch):
    ep = i + 1
    print(f"Epoch {i+1}")
    train_loop(model3,train3_dataloader,optimizer,loss)
    acc = test_loop(model3,test3_dataloader,loss)
    accuracy3.append([ep,acc])
print("Done")

Epoch 1
Batch 0: 0.5873372554779053
Batch 100: 0.04935746267437935
Batch 200: 0.02791978232562542
Batch 300: 0.05095355212688446
Batch 400: 0.0170274805277586
Batch 500: 0.11280377954244614
Batch 600: 0.04813871160149574
Batch 700: 0.03396665304899216
Batch 800: 0.04868549108505249
Batch 900: 0.04061231017112732
Batch 1000: 0.09533291310071945
Batch 1100: 0.15176039934158325
Batch 1200: 0.03058195300400257
Batch 1300: 0.048868339508771896
Batch 1400: 0.06193416193127632
Batch 1500: 0.06630294024944305
Batch 1600: 0.1434815675020218
Batch 1700: 0.018688691779971123
Accuracy: 98.257, loss: 0.07
Epoch 2
Batch 0: 0.053053513169288635
Batch 100: 0.15373097360134125
Batch 200: 0.04397180303931236
Batch 300: 0.01617572084069252
Batch 400: 0.19737963378429413
Batch 500: 0.03426331654191017
Batch 600: 0.0618976354598999
Batch 700: 0.09326892346143723
Batch 800: 0.08059706538915634
Batch 900: 0.058906540274620056
Batch 1000: 0.018257472664117813
Batch 1100: 0.015407927334308624
Batch 1200: 0.028

In [296]:
torch.save(model1.state_dict(),"model1_parameters.pth")
torch.save(model2.state_dict(),"model2_parameters.pth")
torch.save(model3.state_dict(),"model3_parameters.pth")

In [297]:
building1_analysis = pd.DataFrame(accuracy1,columns=['epoch','accuracy'])
building2_analysis = pd.DataFrame(accuracy2,columns=['epoch','accuracy'])
building3_analysis = pd.DataFrame(accuracy3,columns=['epoch','accuracy'])

In [298]:
building1_analysis.to_csv("building1_accuracy.csv",index=False)
building2_analysis.to_csv("building2_accuracy.csv",index=False)
building3_analysis.to_csv("building3_accuracy.csv",index=False)

In [ ]:
building3_analysis

,epoch,accuracy
0,1,98.257
1,2,98.257
2,3,98.257
3,4,98.257
4,5,98.257
5,6,98.257
6,7,98.257
7,8,98.257
8,9,98.257
9,10,98.257


: 

In [91]:
loaded_model = building1_FFN()
loaded_model.state_dict(torch.load("model1_parameters.pth"))
loaded_model.eval()


# input_data = [1,0,1,0,0,0.3,0.5,0.5,0.3,0.1,0.5,0.09,0.5,0.7,0.9]
# input_tensor = torch.FloatTensor(input_data).unsqueeze(0)

# with torch.no_grad():
#     output = loaded_model(input_tensor)

# prediction = output
# print(prediction)
# print(prediction.argmax(dim=1).item())

C:\Users\tousi\AppData\Local\Temp\ipykernel_14492\1081638730.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.state_dict(torch.load("model1_parameters.pth"))

building1_FFN(
  (hidden1): Linear(in_features=32, out_features=20, bias=True)
  (hidden2): Linear(in_features=20, out_features=12, bias=True)
  (hidden3): Linear(in_features=12, out_features=6, bias=True)
  (output): Linear(in_features=6, out_features=2, bias=True)
)

In [92]:
print(df1_encoded.columns)
print(df2_encoded.columns)
print(df3_encoded.columns)

Index(['Fan_status', 'Zone_name_0', 'Zone_name_1', 'Zone_name_2',
       'Zone_name_3', 'Zone_name_4', 'Zone_name_5', 'Zone_name_6',
       'Zone_name_7', 'Zone_name_8', 'Zone_name_9', 'Zone_name_10',
       'Zone_name_11', 'Zone_name_12', 'Zone_name_13', 'Zone_name_14',
       'Zone_name_15', 'Zone_name_16', 'Zone_name_17', 'Zone_name_18',
       'Zone_name_19', 'Zone_name_20', 'Zone_name_21', 'Season_1', 'Season_2',
       'Season_3', 'Season_4', 'Zone_temp', 'Ambient_temp', 'Damper_status',
       'Datetime_diff_mins', 'Zone_temp_diff', 'Ambient_temp_diff'],
      dtype='object')
Index(['Fan_status', 'Zone_name_0', 'Season_1', 'Season_2', 'Season_3',
       'Season_4', 'Zone_temp', 'Slab_temp', 'Dew_temp', 'Ambient_temp',
       'Zone_c02', 'Datetime_diff_mins', 'Zone_temp_diff', 'Slab_temp_diff',
       'Dew_temp_diff', 'Ambient_temp_diff'],
      dtype='object')
Index(['Fan_status', 'Zone_name_0', 'Zone_name_1', 'Zone_name_2',
       'Zone_name_3', 'Zone_name_4', 'Zone_name_5', 'Z

In [93]:
len(df1['Zone_name'].unique())

22

In [94]:
with open("onehot_encoder_building1",'rb') as file:
    building1_encode = pickle.load(file)

In [95]:
with open("minmax_building1","rb") as file:
    building1_minmax = pickle.load(file)

In [108]:
random = np.random.randint(0,100,(1,6))
print(random)

[[46 76 65 24 91 65]]


In [141]:
minmax_data = np.array([3,4,5,6,5,8]).reshape(1,6)
minmax_data = building1_minmax.transform(minmax_data)[0]

c:\Users\tousi\anaconda3\envs\MLenv\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [136]:
print(random_data)

[[ 0.07594937 -0.001544    0.05181347 -0.01801802  0.61023622  0.50466712]]


In [140]:
encode = building1_encode.transform([[1,1]])[0]

c:\Users\tousi\anaconda3\envs\MLenv\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [142]:
combined_data = np.concatenate((encode,minmax_data))

In [120]:
print(len(data))

32


In [143]:
# make prediction
with torch.no_grad():
    pred = loaded_model(torch.FloatTensor(combined_data).unsqueeze(0))
# retrieve index of the value with the highest score
prediction = pred.argmax(dim=1).item()
print(prediction)

1
